In [14]:
import os
import argparse

from rclpy.serialization import deserialize_message, serialize_message
from rosidl_runtime_py.utilities import get_message
import rosbag2_py
from nav_msgs.msg import Odometry
from sensor_msgs.msg import LaserScan

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline


In [25]:
def get_reader(input_bag: str):
    reader = rosbag2_py.SequentialReader()
    reader.open(
        rosbag2_py.StorageOptions(uri=input_bag, storage_id="mcap"),
        rosbag2_py.ConverterOptions(
            input_serialization_format="cdr", output_serialization_format="cdr"
        ),
    )
    return reader

def get_topic_type(topic: str, topic_meta_list: list):
    for topic_meta in topic_meta_list:
        if topic_meta.name == topic:
            return topic_meta.type
    raise ValueError(f"topic {topic} not found in topic_meta_list")

In [31]:
file_path = "/root/ros2_data/20230715_bag/record0_odom_convert_simtime2_0_modified/record0_odom_convert_simtime2_0_modified_0.mcap"
if not os.path.exists(file_path):
    print("File not found: {}".format(file_path))
    exit(1)

reader = get_reader(file_path)
topic_meta_list = reader.get_all_topics_and_types()

topic_name_list = [topic_meta.name for topic_meta in topic_meta_list]

print(topic_name_list)

# load all topics to pandas.DataFrame
# columns: id, topic_name, data, timestamp
msgs_list = []
df = pd.DataFrame(columns=["id", "topic_name", "msg"])
count = 0
while reader.has_next():
    topic, data, t = reader.read_next()

    # convert data to dict
    msg_type = get_message(get_topic_type(topic, topic_meta_list))
    msg = deserialize_message(data, msg_type)

    # df = pd.concat([df, pd.DataFrame([[count, topic, msg]], columns=["id", "topic_name", "msg"])])
    msgs_list.append([count, topic, msg])
    count += 1

df = pd.DataFrame(msgs_list, columns=["id", "topic_name", "msg"])

del reader

df

['/scan_top_lidar', '/scan_front_lidar', '/odom', '/nmea_sentence', '/actual_path']


,id,topic_name,msg
0,0,/scan_top_lidar,sensor_msgs.msg.LaserScan(header=std_msgs.msg....
1,1,/nmea_sentence,nmea_msgs.msg.Sentence(header=std_msgs.msg.Hea...
2,2,/nmea_sentence,nmea_msgs.msg.Sentence(header=std_msgs.msg.Hea...
3,3,/nmea_sentence,nmea_msgs.msg.Sentence(header=std_msgs.msg.Hea...
4,4,/nmea_sentence,nmea_msgs.msg.Sentence(header=std_msgs.msg.Hea...
...,...,...,...
45986,45986,/nmea_sentence,nmea_msgs.msg.Sentence(header=std_msgs.msg.Hea...
45987,45987,/nmea_sentence,nmea_msgs.msg.Sentence(header=std_msgs.msg.Hea...
45988,45988,/nmea_sentence,nmea_msgs.msg.Sentence(header=std_msgs.msg.Hea...
45989,45989,/nmea_sentence,nmea_msgs.msg.Sentence(header=std_msgs.msg.Hea...
